In [ ]:
# ==============================================================================
#  COLORECTAL CANCER TRIAL FAILURE PREDICTION
#  NLP + Machine Learning Pipeline on ClinicalTrials.gov Protocol Text
#
#  Research question:
#    Can NLP applied to colorectal cancer trial protocol text predict
#    trial failure (early termination / low accrual) before recruitment begins?
#
#  Methods:
#    Text preprocessing · TF-IDF · Logistic Regression · Random Forest
#    Gradient Boosting · SVM · SHAP explainability · Latent Semantic Analysis
#    Online (prequential) learning simulation · Calibration · ROC/PR curves
#
#  Data: Real — ClinicalTrials.gov API v2
#
#  References:
#    Pedregosa et al. (2011) JMLR 12:2825-2830 — scikit-learn
#    Lundberg & Lee (2017) NeurIPS — SHAP
#    Devlin et al. (2019) NAACL — BERT
#    Ghadessi et al. (2020) Orphanet J Rare Dis — trial failure predictors
#    Fogel (2018) Contemp Clin Trials Commun — clinical trial success rates
#
#  Author: Sm Hasan ul Bari | MBBS · MSc Biostatistics · MSc Health Economics
#  GitHub: github.com/sm-hasanulbari/colorectal-cancer-nlp-trial-prediction
# ==============================================================================

In [ ]:
# ── CELL 1: IMPORTS ────────────────────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
import os
import re
import json
import shutil
from collections import Counter

warnings.filterwarnings('ignore')

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split, learning_curve)
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve,
                             classification_report, confusion_matrix,
                             brier_score_loss)
import shap

SAVE    = r"C:\Users\drhas\Documents\colorectal-cancer-nlp-trial-prediction\python"
os.makedirs(SAVE, exist_ok=True)

NAVY  = '#003366'; BLUE  = '#0066CC'; RED   = '#C0392B'
GREEN = '#1A7A4A'; AMBER = '#E67E22'; DGRAY = '#555555'
GREY  = '#CCCCCC'; GOLD  = '#C9A84C'; BLACK = '#1A1A1A'

plt.rcParams.update({
    'font.family': 'Georgia', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#CCCCCC', 'axes.linewidth': 0.8,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'xtick.color': DGRAY, 'ytick.color': DGRAY,
    'grid.color': '#EBEBEB', 'grid.linewidth': 0.6
})

def add_title(fig, title, subtitle=''):
    fig.text(0.5, 0.97, title, ha='center', fontsize=14,
             fontweight='bold', color=NAVY)
    if subtitle:
        fig.text(0.5, 0.932, subtitle, ha='center', fontsize=10, color=DGRAY)

def style_ax(ax):
    ax.set_axisbelow(True)
    for sp in ['left', 'bottom']:
        ax.spines[sp].set_color('#CCCCCC')

def source_note(fig, txt):
    fig.text(0.01, 0.01, txt, fontsize=7.5, color='#AAAAAA', va='bottom')

np.random.seed(2025)
print("✓ Cell 1: Imports complete")

In [ ]:
# ── CELL 2: FETCH COLORECTAL CANCER TRIALS ─────────────────────────────────────
def fetch_crc_trials():
    url      = "https://clinicaltrials.gov/api/v2/studies"
    studies  = []
    token    = None
    page_num = 1

    while True:
        params = {
            "query.cond":          "Colorectal Cancer",
            "filter.overallStatus": "TERMINATED,COMPLETED",
            "pageSize":            1000,
            "format":              "json",
            "fields": (
                "NCTId,BriefTitle,OfficialTitle,BriefSummary,"
                "EligibilityCriteria,PrimaryOutcomeMeasure,"
                "StudyType,Phase,OverallStatus,EnrollmentCount,"
                "StartDate,CompletionDate,Condition,InterventionType,"
                "SponsorClass,LeadSponsorName,WhyStopped"
            )
        }
        if token:
            params["pageToken"] = token

        r = requests.get(url, params=params, timeout=60)
        if r.status_code != 200:
            print(f"  ✗ Error page {page_num}: {r.status_code}")
            break

        data    = r.json()
        batch   = data.get('studies', [])
        studies.extend(batch)
        print(f"  Page {page_num}: {len(batch)} trials | Total: {len(studies)}")

        token = data.get('nextPageToken')
        if not token:
            print(f"\n✓ Fetch complete: {len(studies):,} CRC trials")
            break
        page_num += 1

    return studies

print("Fetching colorectal cancer trials from ClinicalTrials.gov...")
raw_studies = fetch_crc_trials()

In [ ]:
# ── CELL 3: EXTRACT FIELDS ─────────────────────────────────────────────────────
def extract_field(study, *keys):
    obj = study
    for k in keys:
        if isinstance(obj, dict):
            obj = obj.get(k, '')
        else:
            return ''
    return str(obj) if obj else ''

def extract_list_field(study, *keys):
    obj = study
    for k in keys[:-1]:
        if isinstance(obj, dict):
            obj = obj.get(k, {})
        else:
            return ''
    last = keys[-1]
    val  = obj.get(last, []) if isinstance(obj, dict) else []
    if isinstance(val, list):
        return ' | '.join([str(v) for v in val[:3]])
    return str(val)

rows = []
for s in raw_studies:
    p = s.get('protocolSection', {})
    rows.append({
        'nct_id':         extract_field(s, 'protocolSection', 'identificationModule', 'nctId'),
        'title':          extract_field(s, 'protocolSection', 'identificationModule', 'briefTitle'),
        'official_title': extract_field(s, 'protocolSection', 'identificationModule', 'officialTitle'),
        'brief_summary':  extract_field(s, 'protocolSection', 'descriptionModule', 'briefSummary'),
        'eligibility':    extract_field(s, 'protocolSection', 'eligibilityModule', 'eligibilityCriteria'),
        'primary_outcome':extract_list_field(s, 'protocolSection', 'outcomesModule', 'primaryOutcomes'),
        'phase':          extract_list_field(s, 'protocolSection', 'designModule', 'phases'),
        'status':         extract_field(s, 'protocolSection', 'statusModule', 'overallStatus'),
        'enrollment':     extract_field(s, 'protocolSection', 'designModule', 'enrollmentInfo', 'count'),
        'sponsor_class':  extract_field(s, 'protocolSection', 'sponsorCollaboratorsModule', 'leadSponsor', 'class'),
        'why_stopped':    extract_field(s, 'protocolSection', 'statusModule', 'whyStopped'),
        'start_year':     extract_field(s, 'protocolSection', 'statusModule', 'startDateStruct', 'date'),
    })

df = pd.DataFrame(rows)
df['enrollment'] = pd.to_numeric(df['enrollment'], errors='coerce')
df['start_year'] = pd.to_numeric(df['start_year'].str[:4], errors='coerce')

print(f"\n✓ Extracted {len(df):,} trials")
print(f"  TERMINATED: {(df['status']=='TERMINATED').sum():,}")
print(f"  COMPLETED:  {(df['status']=='COMPLETED').sum():,}")

In [ ]:
# ── CELL 4: TARGET VARIABLE + TEXT ASSEMBLY ────────────────────────────────────
# Binary target: TERMINATED = 1 (failure), COMPLETED = 0 (success)
df = df[df['status'].isin(['TERMINATED', 'COMPLETED'])].copy()
df['failed'] = (df['status'] == 'TERMINATED').astype(int)

# Assemble protocol text from all available fields
df['protocol_text'] = (
    df['title'].fillna('') + ' ' +
    df['official_title'].fillna('') + ' ' +
    df['brief_summary'].fillna('') + ' ' +
    df['eligibility'].fillna('') + ' ' +
    df['primary_outcome'].fillna('')
)

# Text length as meta-feature
df['text_length']   = df['protocol_text'].str.len()
df['title_length']  = df['title'].str.len()
df['has_phase']     = (df['phase'].str.len() > 0).astype(int)
df['log_enrollment'] = np.log1p(df['enrollment'].fillna(0))

# Remove near-empty texts
df = df[df['text_length'] > 100].copy()
df = df.reset_index(drop=True)

failure_rate = df['failed'].mean()
print(f"\n✓ Modelling dataset: {len(df):,} trials")
print(f"  Failure rate (TERMINATED): {failure_rate*100:.1f}%")
print(f"  Median text length: {df['text_length'].median():.0f} chars")
print(f"  Median enrollment:  {df['enrollment'].median():.0f}")

# Save raw dataset
df.to_csv(os.path.join(SAVE, 'crc_trials_raw.csv'), index=False)
print(f"  Saved: crc_trials_raw.csv")

In [ ]:
# ── CELL 5: TEXT PREPROCESSING + TF-IDF ───────────────────────────────────────
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['protocol_text'].apply(clean_text)

# Medical stop words to remove (too generic to signal failure)
med_stop = [
    'patient', 'patients', 'study', 'trial', 'cancer', 'colorectal',
    'treatment', 'clinical', 'criteria', 'inclusion', 'exclusion',
    'subject', 'subjects', 'arm', 'randomized', 'phase', 'primary',
    'secondary', 'endpoint', 'efficacy', 'safety', 'dose', 'week', 'month',
    'year', 'day', 'least', 'must', 'able', 'prior', 'known', 'history'
]
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
stop_words = list(ENGLISH_STOP_WORDS) + med_stop

# TF-IDF feature extraction
tfidf = TfidfVectorizer(
    max_features    = 5000,
    ngram_range     = (1, 2),
    min_df          = 3,
    max_df          = 0.90,
    sublinear_tf    = True,
    stop_words      = stop_words
)

X_text  = tfidf.fit_transform(df['clean_text'])
y       = df['failed'].values
feature_names = tfidf.get_feature_names_out()

print(f"\n✓ TF-IDF matrix: {X_text.shape[0]:,} docs × {X_text.shape[1]:,} features")
print(f"  Top unigrams: {', '.join(feature_names[:15])}")

# Top terms in failed vs completed trials
df_terms = pd.DataFrame(X_text.toarray(), columns=feature_names)
df_terms['failed'] = y

top_failed    = df_terms[df_terms['failed']==1].drop('failed',axis=1).mean().nlargest(20)
top_completed = df_terms[df_terms['failed']==0].drop('failed',axis=1).mean().nlargest(20)

term_summary = pd.DataFrame({
    'term':       list(top_failed.index) + list(top_completed.index),
    'mean_tfidf': list(top_failed.values) + list(top_completed.values),
    'group':      ['Failed']*len(top_failed) + ['Completed']*len(top_completed)
})
term_summary.to_csv(os.path.join(SAVE, 'crc_top_terms.csv'), index=False)
print("  Saved: crc_top_terms.csv")

In [ ]:
# ── CELL 6: ML CLASSIFIERS ─────────────────────────────────────────────────────
# Train / test split — stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X_text, y, test_size=0.20, random_state=42, stratify=y)

classifiers = {
    'Logistic Regression': CalibratedClassifierCV(
        LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced', random_state=42),
        cv=5, method='isotonic'),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=42),
    'SVM (linear)': CalibratedClassifierCV(
        SVC(kernel='linear', C=0.5, class_weight='balanced', random_state=42),
        cv=5, method='isotonic')
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

print("\nTraining classifiers (5-fold CV)...")
for name, clf in classifiers.items():
    auc_cv  = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='roc_auc',       n_jobs=-1)
    ap_cv   = cross_val_score(clf, X_tr, y_tr, cv=cv, scoring='average_precision', n_jobs=-1)
    # Fit on full training set for test eval
    clf.fit(X_tr, y_tr)
    y_prob  = clf.predict_proba(X_te)[:, 1]
    y_pred  = clf.predict(X_te)
    results.append({
        'Classifier':   name,
        'CV AUC (mean)': round(auc_cv.mean(), 3),
        'CV AUC (SD)':  round(auc_cv.std(),  3),
        'CV AP (mean)': round(ap_cv.mean(),  3),
        'Test AUC':     round(roc_auc_score(y_te, y_prob), 3),
        'Test AP':      round(average_precision_score(y_te, y_prob), 3),
        'Brier Score':  round(brier_score_loss(y_te, y_prob), 3)
    })
    print(f"  {name}: CV AUC={auc_cv.mean():.3f}±{auc_cv.std():.3f} | "
          f"Test AUC={roc_auc_score(y_te, y_prob):.3f}")

df_results = pd.DataFrame(results)
df_results.to_csv(os.path.join(SAVE, 'crc_classifier_results.csv'), index=False)
print("\n✓ Results saved: crc_classifier_results.csv")

# Best classifier for downstream analysis
best_name = df_results.loc[df_results['Test AUC'].idxmax(), 'Classifier']
best_clf  = classifiers[best_name]
print(f"  Best classifier: {best_name}")

# ROC + PR data for all classifiers
roc_rows = []; pr_rows = []
for name, clf in classifiers.items():
    yp = clf.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, yp)
    pre, rec, _ = precision_recall_curve(y_te, yp)
    for f, t in zip(fpr, tpr):
        roc_rows.append({'Classifier': name, 'FPR': round(float(f),4), 'TPR': round(float(t),4)})
    for p, r in zip(pre, rec):
        pr_rows.append({'Classifier': name, 'Precision': round(float(p),4), 'Recall': round(float(r),4)})

pd.DataFrame(roc_rows).to_csv(os.path.join(SAVE, 'crc_roc_curves.csv'), index=False)
pd.DataFrame(pr_rows).to_csv(os.path.join(SAVE, 'crc_pr_curves.csv'), index=False)
print("  Saved: crc_roc_curves.csv, crc_pr_curves.csv")

In [ ]:
# ── CELL 7: SHAP EXPLAINABILITY ────────────────────────────────────────────────
print("\nComputing SHAP values (Logistic Regression)...")

# Use LR for SHAP — interpretable coefficients + LinearExplainer
lr_clf = classifiers['Logistic Regression']
# Get the inner LR estimator for SHAP
try:
    inner_lr = lr_clf.calibrated_classifiers_[0].estimator
except Exception:
    inner_lr = LogisticRegression(C=1.0, max_iter=1000,
                                   class_weight='balanced', random_state=42)
    inner_lr.fit(X_tr, y_tr)

# SHAP LinearExplainer on dense sample
X_sample  = X_te[:500]
explainer = shap.LinearExplainer(inner_lr, X_tr,
                                  feature_perturbation='interventional')
shap_vals = explainer.shap_values(X_sample)

# Top features by mean |SHAP|
mean_shap  = np.abs(shap_vals).mean(axis=0)
top_idx    = np.argsort(mean_shap)[::-1][:30]
top_feats  = feature_names[top_idx]
top_shap   = mean_shap[top_idx]

# SHAP direction: positive = predicts failure
shap_dir   = np.sign(np.array(shap_vals).mean(axis=0))[top_idx]

df_shap = pd.DataFrame({
    'feature':    top_feats,
    'mean_shap':  np.round(top_shap, 5),
    'direction':  ['Failure' if d > 0 else 'Success' for d in shap_dir],
    'rank':       range(1, len(top_feats)+1)
})
df_shap.to_csv(os.path.join(SAVE, 'crc_shap_values.csv'), index=False)
print(f"  Top SHAP feature: {top_feats[0]} (mean |SHAP|={top_shap[0]:.4f})")
print("  Saved: crc_shap_values.csv")

In [ ]:
# ── CELL 8: LATENT SEMANTIC ANALYSIS (LSA proxy for BERT embeddings) ───────────
# LSA via TruncatedSVD on TF-IDF matrix = dense semantic representation
# This is the computationally efficient approximation to dense transformer
# embeddings suitable for CPU-only environments.
# Full BERT fine-tuning would require GPU (HuggingFace transformers).
print("\nComputing LSA semantic embeddings (BERT-proxy, CPU-safe)...")

svd = TruncatedSVD(n_components=100, random_state=42)
X_lsa_tr = svd.fit_transform(X_tr)
X_lsa_te = svd.transform(X_te)

explained_var = svd.explained_variance_ratio_.cumsum()

# LSA + LR classifier
from sklearn.linear_model import LogisticRegression as LR_plain
lsa_clf = LR_plain(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
lsa_clf.fit(X_lsa_tr, y_tr)
lsa_proba  = lsa_clf.predict_proba(X_lsa_te)[:, 1]
lsa_auc    = roc_auc_score(y_te, lsa_proba)
tfidf_proba = classifiers['Logistic Regression'].predict_proba(X_te)[:, 1]
tfidf_auc   = roc_auc_score(y_te, tfidf_proba)

emb_comp = pd.DataFrame({
    'Method': ['TF-IDF (sparse)', 'LSA 100d (dense)', 'LSA 50d (dense)'],
    'Dimensions': [X_text.shape[1], 100, 50],
    'Test AUC': [
        round(tfidf_auc, 3),
        round(lsa_auc, 3),
        round(roc_auc_score(y_te,
            LR_plain(C=1.0, max_iter=1000, class_weight='balanced', random_state=42)
            .fit(TruncatedSVD(50, random_state=42).fit_transform(X_tr),y_tr)
            .predict_proba(TruncatedSVD(50, random_state=42)
                           .fit(X_tr).transform(X_te))[:,1]), 3)
    ],
    'Note': [
        'Bag-of-words, interpretable',
        'Semantic compression, dense',
        'Semantic compression, compact'
    ]
})
emb_comp.to_csv(os.path.join(SAVE, 'crc_embedding_comparison.csv'), index=False)

# Explained variance by component
ev_df = pd.DataFrame({
    'component':           range(1, len(svd.explained_variance_ratio_)+1),
    'explained_variance':  np.round(svd.explained_variance_ratio_, 5),
    'cumulative_variance': np.round(explained_var, 5)
})
ev_df.to_csv(os.path.join(SAVE, 'crc_lsa_variance.csv'), index=False)
print(f"  LSA AUC: {lsa_auc:.3f} | TF-IDF AUC: {tfidf_auc:.3f}")
print(f"  LSA 100 components explain {explained_var[99]*100:.1f}% variance")
print("  Saved: crc_embedding_comparison.csv, crc_lsa_variance.csv")

In [ ]:
# ── CELL 9: ONLINE / PREQUENTIAL LEARNING ──────────────────────────────────────
# Simulate prospective deployment: model trained on historical trials,
# evaluated and updated as new trials arrive in temporal order
print("\nRunning online/prequential learning simulation...")

df_online = df[df['start_year'].notna()].sort_values('start_year').reset_index(drop=True)
X_online  = tfidf.transform(df_online['clean_text'])
y_online  = df_online['failed'].values

online_rows = []
min_train   = max(50, int(len(df_online) * 0.10))
window      = 50  # rolling evaluation window

for t in range(min_train, len(y_online), 10):
    clf_ol = LogisticRegression(C=1.0, max_iter=500,
                                 class_weight='balanced', random_state=42)
    clf_ol.fit(X_online[:t], y_online[:t])
    # Evaluate on next window
    end = min(t + window, len(y_online))
    if end <= t:
        break
    proba_window = clf_ol.predict_proba(X_online[t:end])[:, 1]
    if len(np.unique(y_online[t:end])) < 2:
        continue
    online_rows.append({
        'n_train':    t,
        'year_approx': float(df_online['start_year'].iloc[min(t, len(df_online)-1)]),
        'window_auc': round(roc_auc_score(y_online[t:end], proba_window), 3),
        'window_ap':  round(average_precision_score(y_online[t:end], proba_window), 3),
        'failure_rate_window': round(y_online[t:end].mean(), 3)
    })

df_online_res = pd.DataFrame(online_rows)
df_online_res.to_csv(os.path.join(SAVE, 'crc_online_learning.csv'), index=False)
print(f"  Online evaluation points: {len(df_online_res)}")
print(f"  Final window AUC: {df_online_res['window_auc'].iloc[-1]:.3f}")
print("  Saved: crc_online_learning.csv")

In [ ]:
# ── CELL 10: CALIBRATION ───────────────────────────────────────────────────────
calib_rows = []
for name, clf in classifiers.items():
    yp = clf.predict_proba(X_te)[:, 1]
    prob_true, prob_pred = calibration_curve(y_te, yp, n_bins=10)
    brier = brier_score_loss(y_te, yp)
    for pt, pp in zip(prob_true, prob_pred):
        calib_rows.append({
            'Classifier': name,
            'mean_pred_prob': round(float(pp), 4),
            'fraction_pos':   round(float(pt), 4),
            'brier_score':    round(brier, 4)
        })

df_calib = pd.DataFrame(calib_rows)
df_calib.to_csv(os.path.join(SAVE, 'crc_calibration.csv'), index=False)
print("\n✓ Calibration saved: crc_calibration.csv")

In [ ]:
# ── CELL 11: FIGURES ───────────────────────────────────────────────────────────
clf_colors = {
    'Logistic Regression': NAVY,
    'Random Forest':       GREEN,
    'Gradient Boosting':   BLUE,
    'SVM (linear)':        AMBER
}

# ── FIG 1: Data overview ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
add_title(fig, 'Colorectal Cancer Clinical Trials — Dataset Overview',
          f'ClinicalTrials.gov API v2 · N={len(df):,} trials · March 2026')

ax = axes[0]
status_counts = df['status'].value_counts()
bc = [GREEN if s == 'COMPLETED' else RED for s in status_counts.index]
bars = ax.bar(status_counts.index, status_counts.values, color=bc,
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:,}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=10, color=BLACK)
ax.set_title('A — Trial Status', fontsize=11, color=NAVY, pad=10)
ax.set_ylabel('Number of Trials', color=DGRAY)
ax.yaxis.grid(True); style_ax(ax)

ax = axes[1]
yr_counts = df[df['start_year'].between(2000, 2025)].groupby('start_year')['failed'].agg(['sum','count'])
yr_counts['rate'] = yr_counts['sum'] / yr_counts['count']
ax.fill_between(yr_counts.index, yr_counts['rate']*100, color=NAVY, alpha=0.15)
ax.plot(yr_counts.index, yr_counts['rate']*100, color=NAVY, linewidth=2.5)
ax.axhline(failure_rate*100, color=RED, linestyle='--', linewidth=1.5,
           label=f'Overall {failure_rate*100:.1f}%')
ax.set_title('B — Failure Rate by Start Year', fontsize=11, color=NAVY, pad=10)
ax.set_xlabel('Trial Start Year', color=DGRAY)
ax.set_ylabel('Termination Rate (%)', color=DGRAY)
ax.legend(fontsize=9); ax.yaxis.grid(True); style_ax(ax)

ax = axes[2]
sp_counts = df.groupby('sponsor_class')['failed'].agg(['mean','count']).reset_index()
sp_counts = sp_counts[sp_counts['count'] >= 10].sort_values('mean')
bc = [GREEN if m < failure_rate else RED for m in sp_counts['mean']]
ax.barh(sp_counts['sponsor_class'], sp_counts['mean']*100, color=bc, edgecolor='white')
ax.axvline(failure_rate*100, color=DGRAY, linestyle='--', linewidth=1.5)
ax.set_title('C — Failure Rate by Sponsor', fontsize=11, color=NAVY, pad=10)
ax.set_xlabel('Termination Rate (%)', color=DGRAY)
ax.xaxis.grid(True); style_ax(ax)

source_note(fig, 'Real data. Source: ClinicalTrials.gov API v2. '
                 'Ref: Fogel (2018) Contemp Clin Trials Commun.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig1_data_overview.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 1 saved")

# ── FIG 2: TF-IDF top terms ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
add_title(fig, 'TF-IDF Term Analysis — Failed vs Completed Trials',
          'Top 15 discriminating terms by mean TF-IDF score (ngram range 1–2)')

for ax, grp, col, title in zip(
    axes,
    ['Failed', 'Completed'],
    [RED, GREEN],
    ['A — High TF-IDF in TERMINATED Trials', 'B — High TF-IDF in COMPLETED Trials']
):
    sub = term_summary[term_summary['group'] == grp].head(15)
    ax.barh(sub['term'][::-1], sub['mean_tfidf'][::-1], color=col,
            alpha=0.85, edgecolor='white')
    ax.set_title(title, fontsize=11, color=NAVY, pad=10)
    ax.set_xlabel('Mean TF-IDF Score', color=DGRAY)
    ax.xaxis.grid(True); style_ax(ax)

source_note(fig, 'Synthetic-calibrated real data. '
                 'Ref: Pedregosa et al. (2011) JMLR 12:2825.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig2_tfidf_terms.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 2 saved")

# ── FIG 3: ROC + PR curves ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
add_title(fig, 'Classifier Performance — ROC and Precision-Recall Curves',
          '20% hold-out test set · Stratified split')

ax = axes[0]
for name, clf in classifiers.items():
    yp = clf.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, yp)
    auc = roc_auc_score(y_te, yp)
    ax.plot(fpr, tpr, color=clf_colors[name], linewidth=2.2,
            label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', color=DGRAY)
ax.set_ylabel('True Positive Rate', color=DGRAY)
ax.set_title('A — ROC Curves', fontsize=11, color=NAVY, pad=10)
ax.legend(fontsize=9, frameon=True, framealpha=0.95)
ax.yaxis.grid(True); style_ax(ax)

ax = axes[1]
baseline_ap = failure_rate
for name, clf in classifiers.items():
    yp = clf.predict_proba(X_te)[:, 1]
    pre, rec, _ = precision_recall_curve(y_te, yp)
    ap = average_precision_score(y_te, yp)
    ax.plot(rec, pre, color=clf_colors[name], linewidth=2.2,
            label=f'{name} (AP={ap:.3f})')
ax.axhline(baseline_ap, color=GREY, linestyle='--', linewidth=1.2,
           label=f'Baseline ({baseline_ap:.2f})')
ax.set_xlabel('Recall', color=DGRAY)
ax.set_ylabel('Precision', color=DGRAY)
ax.set_title('B — Precision-Recall Curves', fontsize=11, color=NAVY, pad=10)
ax.legend(fontsize=9, frameon=True, framealpha=0.95)
ax.yaxis.grid(True); style_ax(ax)

source_note(fig, 'Ref: Saito & Rehmsmeier (2015) PLOS ONE — PR curves for imbalanced data.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig3_roc_pr.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 3 saved")

# ── FIG 4: SHAP beeswarm-style ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))

top20 = df_shap.head(20).iloc[::-1]
bc    = [RED if d == 'Failure' else GREEN for d in top20['direction']]
bars  = ax.barh(top20['feature'], top20['mean_shap'], color=bc,
                alpha=0.85, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, top20['mean_shap']):
    ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8.5, color=DGRAY)

fail_patch = mpatches.Patch(color=RED,   alpha=0.85, label='Predicts TERMINATED')
succ_patch = mpatches.Patch(color=GREEN, alpha=0.85, label='Predicts COMPLETED')
ax.legend(handles=[fail_patch, succ_patch], fontsize=10, frameon=True)
ax.set_xlabel('Mean |SHAP Value| — Impact on Failure Prediction', color=DGRAY)
ax.xaxis.grid(True); style_ax(ax)

add_title(fig, 'SHAP Feature Importance — Logistic Regression on Protocol Text',
          'Top 20 features by mean absolute SHAP value · Colour = prediction direction')
source_note(fig, 'Ref: Lundberg & Lee (2017) NeurIPS — SHAP. '
                 'Lundberg et al. (2020) Nat Mach Intell — TreeSHAP.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig4_shap.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 4 saved")

# ── FIG 5: Calibration curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
add_title(fig, 'Classifier Calibration — Reliability Diagrams',
          'Perfect calibration = diagonal · Below diagonal = overconfident')

ax = axes[0]
for name, clf in classifiers.items():
    yp = clf.predict_proba(X_te)[:, 1]
    pt, pp = calibration_curve(y_te, yp, n_bins=10)
    ax.plot(pp, pt, 's-', color=clf_colors[name], linewidth=2,
            markersize=7, label=name)
ax.plot([0,1],[0,1], 'k--', linewidth=1.5, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability', color=DGRAY)
ax.set_ylabel('Fraction of Positives (True)', color=DGRAY)
ax.set_title('A — Reliability Diagrams', fontsize=11, color=NAVY, pad=10)
ax.legend(fontsize=9, frameon=True, framealpha=0.95)
ax.yaxis.grid(True); style_ax(ax)

ax = axes[1]
brier_scores = [brier_score_loss(y_te, clf.predict_proba(X_te)[:,1])
                for clf in classifiers.values()]
bc = [GREEN if b < 0.20 else AMBER for b in brier_scores]
bars = ax.bar(list(classifiers.keys()), brier_scores, color=bc, edgecolor='white')
for bar, val in zip(bars, brier_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', fontsize=10, color=BLACK)
ax.axhline(failure_rate*(1-failure_rate), color=RED, linestyle='--',
           linewidth=1.5, label=f'Null model ({failure_rate*(1-failure_rate):.3f})')
ax.set_title('B — Brier Score (lower = better)', fontsize=11, color=NAVY, pad=10)
ax.set_ylabel('Brier Score', color=DGRAY)
ax.tick_params(axis='x', rotation=15)
ax.legend(fontsize=9); ax.yaxis.grid(True); style_ax(ax)

source_note(fig, 'Ref: Brier (1950) Mon Weather Rev — proper scoring rules.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig5_calibration.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 5 saved")

# ── FIG 6: Online learning ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
add_title(fig, 'Online / Prequential Learning — Prospective Deployment Simulation',
          'Model trained on historical trials, evaluated on prospective arrivals (rolling window)')

ax = axes[0]
ax.fill_between(df_online_res['n_train'], df_online_res['window_auc'],
                color=NAVY, alpha=0.10)
ax.plot(df_online_res['n_train'], df_online_res['window_auc'],
        color=NAVY, linewidth=2.5, label='Window AUC')
ax.plot(df_online_res['n_train'], df_online_res['window_ap'],
        color=BLUE, linewidth=2, linestyle='--', label='Window AP')
ax.set_xlabel('Training Set Size (N trials)', color=DGRAY)
ax.set_ylabel('Performance Metric', color=DGRAY)
ax.set_title('A — AUC and AP vs Training Size', fontsize=11, color=NAVY, pad=10)
ax.legend(fontsize=10, frameon=True)
ax.yaxis.grid(True); style_ax(ax)

ax = axes[1]
ax.plot(df_online_res['n_train'], df_online_res['failure_rate_window']*100,
        color=AMBER, linewidth=2.5)
ax.axhline(failure_rate*100, color=RED, linestyle='--', linewidth=1.5,
           label=f'Overall rate {failure_rate*100:.1f}%')
ax.set_xlabel('Training Set Size (N trials)', color=DGRAY)
ax.set_ylabel('Window Failure Rate (%)', color=DGRAY)
ax.set_title('B — Concept Drift Detection (Failure Rate)', fontsize=11, color=NAVY, pad=10)
ax.legend(fontsize=10); ax.yaxis.grid(True); style_ax(ax)

source_note(fig, 'Ref: Gama et al. (2014) ACM Comput Surv — concept drift adaptation. '
                 'Losing et al. (2018) IEEE TKDE — incremental learning survey.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig6_online_learning.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 6 saved")

# ── FIG 7: Embedding comparison + learning curve ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
add_title(fig, 'Representation Methods — TF-IDF vs LSA Semantic Embeddings',
          'LSA via Truncated SVD = computationally efficient BERT proxy for CPU environments')

ax = axes[0]
df_emb = pd.read_csv(os.path.join(SAVE, 'crc_embedding_comparison.csv'))
bc = [NAVY, BLUE, GREEN]
bars = ax.bar(df_emb['Method'], df_emb['Test AUC'], color=bc, edgecolor='white',
              width=0.5)
for bar, val in zip(bars, df_emb['Test AUC']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', fontsize=11, color=BLACK, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_ylabel('Test AUC', color=DGRAY)
ax.set_title('A — AUC by Representation Method', fontsize=11, color=NAVY, pad=10)
ax.tick_params(axis='x', rotation=12)
ax.yaxis.grid(True); style_ax(ax)

ax = axes[1]
lc_sizes = np.linspace(0.1, 0.9, 9)
train_scores = []; val_scores = []
for frac in lc_sizes:
    n = max(20, int(len(y_tr) * frac))
    clf_lc = LogisticRegression(C=1.0, max_iter=500,
                                 class_weight='balanced', random_state=42)
    clf_lc.fit(X_tr[:n], y_tr[:n])
    train_scores.append(roc_auc_score(y_tr[:n], clf_lc.predict_proba(X_tr[:n])[:,1]))
    val_scores.append(roc_auc_score(y_te, clf_lc.predict_proba(X_te)[:,1]))
ns = [max(20, int(len(y_tr)*f)) for f in lc_sizes]
ax.plot(ns, train_scores, 'o-', color=NAVY,  linewidth=2.5, label='Train AUC')
ax.plot(ns, val_scores,   's-', color=GREEN, linewidth=2.5, label='Validation AUC')
ax.fill_between(ns, train_scores, val_scores, alpha=0.08, color=AMBER,
                label='Generalisation gap')
ax.set_xlabel('Training Set Size', color=DGRAY)
ax.set_ylabel('ROC-AUC', color=DGRAY)
ax.set_title('B — Learning Curve (Logistic Regression)', fontsize=11, color=NAVY, pad=10)
ax.legend(fontsize=9, frameon=True)
ax.yaxis.grid(True); style_ax(ax)

source_note(fig, 'LSA ref: Deerwester et al. (1990) JASIST. '
                 'BERT ref: Devlin et al. (2019) NAACL — for full transformer fine-tuning.')
plt.tight_layout(rect=[0, 0.03, 1, 0.91])
plt.savefig(os.path.join(SAVE, 'crc_fig7_embeddings_learning.png'),
            dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Figure 7 saved")

In [ ]:
# ── CELL 12: KPI SUMMARY CSV ───────────────────────────────────────────────────
best_row = df_results[df_results['Classifier'] == best_name].iloc[0]

kpi = pd.DataFrame({
    'Metric': [
        'Total CRC trials',
        'Terminated (failed)',
        'Completed (success)',
        'Overall failure rate',
        'Best classifier',
        'Best test AUC',
        'Best CV AUC',
        'Best test AP',
        'Top SHAP feature',
        'TF-IDF features',
        'LSA components',
        'Online final AUC'
    ],
    'Value': [
        len(df),
        int((df['failed']==1).sum()),
        int((df['failed']==0).sum()),
        round(failure_rate, 4),
        best_name,
        best_row['Test AUC'],
        best_row['CV AUC (mean)'],
        best_row['Test AP'],
        top_feats[0],
        X_text.shape[1],
        100,
        round(df_online_res['window_auc'].iloc[-1], 3)
    ]
})
kpi.to_csv(os.path.join(SAVE, 'crc_kpi.csv'), index=False)
df_results.to_csv(os.path.join(SAVE, 'crc_classifier_results.csv'), index=False)

print("\n" + "="*65)
print("✓ NOTEBOOK COMPLETE — CRC NLP/ML Trial Failure Prediction")
print(f"  Trials: {len(df):,} | Failure rate: {failure_rate*100:.1f}%")
print(f"  Best classifier: {best_name} | AUC: {best_row['Test AUC']:.3f}")
print(f"  TF-IDF features: {X_text.shape[1]:,} | Top: {top_feats[0]}")
print(f"  7 figures + 9 CSVs saved to: {SAVE}")
print("="*65)